# LAB 2: Docling — PDFs Escaneados e OCR
## Aula 5 — Docling e Ingestão Inteligente de Documentos
### MBA em RAG & CAG Aplicados a Direito e Segurança Pública

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

**⏱️ Duração estimada:** 50 minutos  
**📚 Pré-requisitos:** Lab 1 concluído (instalação e configuração básica do Docling)  
**🎯 Objetivos:**
- Configurar o pipeline OCR do Docling para documentos escaneados
- Criar um laudo pericial escaneado simulado (imagem → PDF)
- Comparar a qualidade do OCR entre diferentes engines (EasyOCR vs. Tesseract)
- Implementar detecção automática de documentos que precisam de OCR
- Processar um relatório de inteligência com estrutura complexa

## 📋 Contexto Jurídico

Laudos periciais do Instituto de Criminalística, boletins de ocorrência antigos
e certidões cartoriais frequentemente chegam como imagens escaneadas.
Um sistema RAG jurídico robusto precisa processar esses documentos fielmente,
pois dados periciais incorretamente extraídos podem comprometer toda a análise.

In [ ]:
# Instalação — engines OCR adicionais
!pip install docling>=2.0.0 --quiet
!pip install easyocr pillow reportlab pdfplumber --quiet
!apt-get install -y tesseract-ocr tesseract-ocr-por -qq  # Tesseract com português
print("✅ Instalação concluída!")

In [ ]:
import sys, warnings, os, json, re
from pathlib import Path
warnings.filterwarnings('ignore')

import docling, pdfplumber
from PIL import Image, ImageDraw, ImageFont
import io

print("✅ Ambiente configurado")
print(f"Python {sys.version.split()[0]}")

## 📄 Etapa 1: Criar Laudo Pericial Escaneado (Simulação)

Vamos criar uma imagem PNG que simula um laudo pericial escaneado e
convertê-la em PDF para testar o OCR do Docling.

**⏱️ Tempo estimado: 8 minutos**

In [ ]:
from PIL import Image, ImageDraw
import reportlab.pdfgen.canvas as canvas
from reportlab.lib.pagesizes import A4

# Criar imagem simulando laudo escaneado
IMG_W, IMG_H = 2480, 3508  # A4 a 300 DPI
img = Image.new('RGB', (IMG_W, IMG_H), color='white')
draw = ImageDraw.Draw(img)

# Simular texto do laudo (sem fontes especiais para compatibilidade Colab)
linhas = [
    (100, 80, 'INSTITUTO DE CRIMINALISTICA - SP'),
    (100, 130, 'LAUDO DE EXAME DE SUBSTANCIA ENTORPECENTE'),
    (100, 180, 'Numero: IC-SP-2024-045821'),
    (100, 230, 'Data: 20/02/2024'),
    (100, 280, '-------------------------------------------'),
    (100, 330, 'SOLICITANTE: 15a Delegacia de Policia - SP'),
    (100, 380, 'BOLETIM DE OCORRENCIA: 001234-2024'),
    (100, 430, '-------------------------------------------'),
    (100, 480, 'OBJETO: Material branco amorfo em 3 embalagens'),
    (100, 530, '-------------------------------------------'),
    (100, 580, 'RESULTADOS:'),
    (100, 630, 'AMOSTRA 1 - Massa: 87.3g - POSITIVO Cloridrato Cocaina'),
    (100, 680, 'AMOSTRA 2 - Massa: 143.7g - POSITIVO Cloridrato Cocaina'),
    (100, 730, 'AMOSTRA 3 - Massa: 52.8g - POSITIVO Cloridrato Cocaina'),
    (100, 780, '-------------------------------------------'),
    (100, 830, 'CONCLUSAO: Substancias sao Cloridrato de Cocaina'),
    (100, 880, 'conforme Portaria SVS/MS 344/98, art. 33 Lei 11343/2006'),
    (100, 930, '-------------------------------------------'),
    (100, 980, 'Sao Paulo, 20 de fevereiro de 2024'),
    (100, 1030, 'PERITO CRIMINAL: Dr. Paulo Roberto Ferreira'),
    (100, 1080, 'CRQ-SP 123456'),
]

# Desenhar texto (fonte padrão PIL)
for x, y, texto in linhas:
    draw.text((x, y), texto, fill='black')

# Adicionar 'ruído' de escaneamento
import random
for _ in range(500):
    x = random.randint(0, IMG_W-1)
    y = random.randint(0, IMG_H-1)
    draw.point((x, y), fill=(200, 200, 200))

# Salvar como PNG
CAMINHO_IMG = '/tmp/laudo_escaneado.png'
img.save(CAMINHO_IMG)

# Converter imagem em PDF (usando reportlab)
CAMINHO_PDF_SCAN = '/tmp/laudo_escaneado.pdf'
c = canvas.Canvas(CAMINHO_PDF_SCAN, pagesize=A4)
w, h = A4
c.drawImage(CAMINHO_IMG, 0, 0, width=w, height=h)
c.save()

print(f"✅ Imagem criada: {CAMINHO_IMG}")
print(f"✅ PDF escaneado criado: {CAMINHO_PDF_SCAN}")
print(f"   Tamanho: {os.path.getsize(CAMINHO_PDF_SCAN)/1024:.1f} KB")

## 🔍 Etapa 2: Detectar Automaticamente se o PDF Precisa de OCR

Antes de processar com OCR (mais lento), verificamos se o PDF já tem texto extraível.

**⏱️ Tempo estimado: 5 minutos**

In [ ]:
def precisa_ocr(caminho_pdf, paginas_amostra=3, threshold=100):
    # Verifica se PDF precisa de OCR baseado na quantidade de texto extraivel
    total_chars = 0
    try:
        with pdfplumber.open(caminho_pdf) as pdf:
            for i, pagina in enumerate(pdf.pages[:paginas_amostra]):
                texto = pagina.extract_text() or ''
                total_chars += len(texto.strip())
    except Exception as e:
        print(f'Erro ao analisar PDF: {e}')
        return True

    media_chars = total_chars / min(paginas_amostra, 1)
    precisa = media_chars < threshold

    status = 'SIM (escaneado)' if precisa else 'NAO (texto nativo)'
    print(f"Media de chars por pagina: {media_chars:.0f}")
    print(f"Precisa OCR: {status}")
    return precisa

# Testar com nossos PDFs
print("=== Laudo ESCANEADO ===")
r1 = precisa_ocr(CAMINHO_PDF_SCAN)

# Comparar com PDF nativo (do Lab 1)
CAMINHO_PDF_NATIVO = '/tmp/acordao_stj_exemplo.pdf'
if Path(CAMINHO_PDF_NATIVO).exists():
    print("\n=== Acordao NATIVO ===")
    r2 = precisa_ocr(CAMINHO_PDF_NATIVO)
else:
    print("\n(PDF nativo do Lab 1 nao disponivel - execute o Lab 1 primeiro)")

## ⚙️ Etapa 3: Configurar Pipeline OCR do Docling

O Docling suporta múltiplos engines OCR. Vamos configurar e testar cada um.

**⏱️ Tempo estimado: 15 minutos**

In [ ]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    EasyOcrOptions,
)

# Pipeline com EasyOCR
ocr_easyocr = EasyOcrOptions(
    lang=['pt', 'en'],
    gpu=False,
)

pipeline_ocr = PdfPipelineOptions(
    do_ocr=True,
    do_table_structure=True,
    ocr_options=ocr_easyocr,
)

converter_ocr = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_ocr)
    }
)

print("⏳ Processando laudo escaneado com OCR (EasyOCR)...")
print("   (Pode demorar 30-60s - EasyOCR carrega modelo de ML)")

try:
    resultado_ocr = converter_ocr.convert(CAMINHO_PDF_SCAN)
    texto_ocr = resultado_ocr.document.export_to_markdown()
    print(f"\n✅ OCR concluido: {len(texto_ocr)} caracteres extraidos")
    print("\n--- TEXTO EXTRAIDO PELO OCR (primeiros 800 chars) ---")
    print(texto_ocr[:800])
except Exception as e:
    print(f"⚠️ Erro no OCR: {e}")
    print("Fallback: usando conversao padrao...")
    converter_padrao = DocumentConverter()
    resultado_ocr = converter_padrao.convert(CAMINHO_PDF_SCAN)
    texto_ocr = resultado_ocr.document.export_to_markdown()
    print(f"Texto extraido (sem OCR): {len(texto_ocr)} chars")

In [ ]:
# Analise qualitativa do OCR
print("=== ANALISE DO OCR ===")

# Verificar se termos juridicos foram reconhecidos
termos_esperados = [
    'LAUDO', 'INSTITUTO', 'CRIMINALISTICA', 'COCAINA', 'AMOSTRA'
]

texto_upper = texto_ocr.upper()
print("\nTermos juridicos reconhecidos:")
for termo in termos_esperados:
    encontrado = termo.upper() in texto_upper
    emoji = '✅' if encontrado else '❌'
    print(f"  {emoji} {termo}")

# Contar numeros extraidos (importantes em laudos)
import re
numeros = re.findall(r'\d+[,.]?\d*', texto_ocr)
print(f"\nNúmeros extraidos: {len(numeros)}")
print(f"Exemplos: {numeros[:10]}")

## 🔄 Etapa 4: Pipeline Automático (Nativo vs. OCR)

Na prática, um sistema de ingestão deve decidir automaticamente quando usar OCR.
Vamos implementar uma função que faz isso de forma transparente.

**⏱️ Tempo estimado: 10 minutos**

In [ ]:
def converter_documento_auto(caminho_pdf, threshold_ocr=100):
    # Converte documento automaticamente: nativo ou OCR conforme necessidade
    print(f'Processando: {Path(caminho_pdf).name}')

    usa_ocr = precisa_ocr(caminho_pdf, threshold=threshold_ocr)

    if usa_ocr:
        print('→ Usando pipeline OCR (EasyOCR)')
        try:
            ocr_opts = EasyOcrOptions(lang=['pt', 'en'], gpu=False)
            pipe_opts = PdfPipelineOptions(do_ocr=True, ocr_options=ocr_opts)
            conv = DocumentConverter(format_options={
                InputFormat.PDF: PdfFormatOption(pipeline_options=pipe_opts)
            })
        except Exception:
            conv = DocumentConverter()
    else:
        print('→ Usando pipeline nativo (sem OCR)')
        conv = DocumentConverter()

    resultado = conv.convert(caminho_pdf)
    doc = resultado.document
    markdown = doc.export_to_markdown()

    return {
        'markdown': markdown,
        'doc': doc,
        'usou_ocr': usa_ocr,
        'num_chars': len(markdown),
        'num_paginas': len(doc.pages),
        'arquivo': Path(caminho_pdf).name
    }

print("\n=== TESTE DO CONVERSOR AUTOMATICO ===")

# Testar com laudo escaneado
res_scan = converter_documento_auto(CAMINHO_PDF_SCAN)
print(f"\n✅ Resultado do laudo escaneado:")
print(f"   • OCR usado: {res_scan['usou_ocr']}")
print(f"   • Chars extraidos: {res_scan['num_chars']}")
print(f"   • Paginas: {res_scan['num_paginas']}")

## ✅ Checkpoint — Lab 2

Execute a célula abaixo para verificar se os objetivos foram atingidos.

In [ ]:
print("=" * 60)
print("CHECKPOINT — LAB 2")
print("=" * 60)
erros = []

# 1. PDF escaneado criado?
if not Path(CAMINHO_PDF_SCAN).exists():
    erros.append('❌ PDF escaneado não criado')
else:
    print("✅ PDF escaneado criado")

# 2. Detecção de OCR funcionando?
try:
    resultado_deteccao = precisa_ocr(CAMINHO_PDF_SCAN)
    if resultado_deteccao:
        print("✅ Detecção de OCR funcionando corretamente")
    else:
        print("⚠️ Detecção identificou como nativo (esperado escaneado)")
except Exception as e:
    erros.append(f"❌ Erro na detecção: {e}")

# 3. OCR extraiu texto?
if len(texto_ocr) < 50:
    erros.append('❌ OCR extraiu muito pouco texto')
else:
    print(f"✅ OCR extraiu {len(texto_ocr)} chars do laudo")

# 4. Conversor automático funcionando?
if 'markdown' in res_scan and len(res_scan['markdown']) > 0:
    print("✅ Conversor automático funcionando")
else:
    erros.append('❌ Conversor automático falhou')

if erros:
    for e in erros: print(e)
else:
    print("\n🎉 TODOS OS OBJETIVOS ATINGIDOS! Avance para o Lab 3.")

## 🏋️ Exercício

**Desafio:** Crie um PDF escaneado com mais ruído (aumente o número de pontos
de 500 para 2000 no código da Etapa 1) e avalie como isso afeta a qualidade do OCR.

Métricas para avaliar:
1. Quantos dos 5 termos jurídicos são reconhecidos?
2. Os números (massas das amostras) são extraídos corretamente?
3. Qual é a diferença em chars extraídos?


In [ ]:
# 🏋️ EXERCÍCIO: Experimente com mais ruído no documento
# Modifique o range(500) para range(2000) ou range(5000)
# e execute o pipeline OCR novamente para comparar a qualidade

# Sua solução aqui:
print("💭 Implemente aqui seu experimento com diferentes níveis de ruído")

## 📖 Referências (ABNT)

AUER, Peter et al. **Docling Technical Report**. arXiv:2408.09869, 2024.

BAEK, J. et al. **Character Region Awareness for Text Detection (CRAFT)**. CVPR, 2019.
Disponível em: <https://github.com/JaidedAI/EasyOCR>. Acesso em: abr. 2026.

SMITH, Ray. **An Overview of the Tesseract OCR Engine**. ICDAR, IEEE, 2007.
